# vector-normalize-keepdim — worked example 1: L2-normalize a 3D point cloud batch

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `vector-normalize-keepdim`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

To unit-normalize a batch of vectors, divide each vector by its own L2 norm. Using `.norm(dim=-1, keepdim=True)` keeps the trailing dimension so the shape stays compatible for broadcast division: a `(B, D)` tensor divided by a `(B, 1)` norm tensor works elementwise across all D components.

## Worked solution

**Step 1 — compute per-row norms.**
We call `x.norm(dim=-1, keepdim=True)`. `dim=-1` collapses the last axis (the spatial coordinates), giving shape `(B, 1)`. The `keepdim=True` is essential: without it the result would be shape `(B,)`, which would broadcast incorrectly against `(B, D)`.

**Step 2 — divide to normalize.**
We divide `x` by the norms. Because norms are `(B, 1)` and `x` is `(B, D)`, broadcasting extends each scalar norm across all D coordinates of its row. Each row of the result lies on the unit sphere.

**Step 3 — verify.**
We compute `.norm(dim=-1)` on the output and confirm every entry equals 1.0 (within floating-point tolerance).

In [ ]:
import torch as t

t.manual_seed(42)
B, D = 8, 3  # batch of 8 three-dimensional points

x = t.randn(B, D)

# Step 1: per-row L2 norms, keepdim so shape is (B, 1) not (B,)
norms = x.norm(dim=-1, keepdim=True)   # (8, 1)

# Step 2: broadcast divide
unit_x = x / norms                      # (8, 3)

# Verify every row has norm ~1
row_norms = unit_x.norm(dim=-1)
print('Row norms (all should be ~1.0):', row_norms.tolist())
print('Max deviation from 1:', (row_norms - 1.0).abs().max().item())